# CUMULUS — Pipeline-Level Evaluation

This notebook reconstructs the pipeline-level ranking from the two CUMULUS result workbooks.

For each complete pipeline configuration (`imputation → outlier handling → normalization`):

1. the KS statistic is averaged across the four signal features within each dataset;
2. the two dataset-level KS values (Cheating and D2) are averaged with equal weight;
3. pipelines are ranked in ascending order of the overall KS value (lower is better).

The resulting ranking is exported to `CUMULUS_pipeline_ranking.csv`.


In [1]:

from pathlib import Path
from collections import defaultdict
from statistics import mean
from html import escape
from IPython.display import display, HTML
import csv
import re
import zipfile
import xml.etree.ElementTree as ET

BASE = Path.cwd()

def resolve_input(pattern):
    """Resolve exactly one input workbook from the notebook directory."""
    matches = sorted(BASE.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file found for pattern: {pattern}")
    if len(matches) > 1:
        # Prefer the newest numbered/current variant if several copies are present.
        matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0]

CHEATING_FILE = resolve_input("CUMULUS_Output_Cheating*.xlsx")
D2_FILE = resolve_input("CUMULUS_Output_d2*.xlsx")

print(f"Cheating input: {CHEATING_FILE.name}")
print(f"D2 input:       {D2_FILE.name}")


Cheating input: CUMULUS_Output_Cheating(3).xlsx
D2 input:       CUMULUS_Output_d2(3).xlsx


In [2]:

# Minimal XLSX reader for one worksheet.
# This keeps the notebook dependency-light: only Python's standard library is
# needed for reading the two workbooks.

NS_MAIN = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
NS_REL = {"r": "http://schemas.openxmlformats.org/package/2006/relationships"}
NS_OFFICE_REL = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

def _column_index(cell_ref):
    letters = re.match(r"([A-Z]+)", cell_ref).group(1)
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord("A") + 1)
    return idx - 1

def _shared_strings(zf):
    try:
        xml = zf.read("xl/sharedStrings.xml")
    except KeyError:
        return []
    root = ET.fromstring(xml)
    strings = []
    for si in root.findall("x:si", NS_MAIN):
        parts = [t.text or "" for t in si.iter("{%s}t" % NS_MAIN["x"])]
        strings.append("".join(parts))
    return strings

def _worksheet_path(zf, sheet_name):
    workbook = ET.fromstring(zf.read("xl/workbook.xml"))
    rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))

    rel_map = {
        rel.attrib["Id"]: rel.attrib["Target"]
        for rel in rels.findall("r:Relationship", NS_REL)
    }

    sheets = workbook.find("x:sheets", NS_MAIN)
    for sheet in sheets:
        if sheet.attrib.get("name") == sheet_name:
            rel_id = sheet.attrib.get("{%s}id" % NS_OFFICE_REL)
            target = rel_map[rel_id].lstrip("/")
            if target.startswith("xl/"):
                return target
            return "xl/" + target
    raise KeyError(f"Worksheet not found: {sheet_name}")

def read_xlsx_sheet(path, sheet_name):
    """Read an XLSX worksheet into a list of row dictionaries."""
    with zipfile.ZipFile(path) as zf:
        shared = _shared_strings(zf)
        sheet_path = _worksheet_path(zf, sheet_name)
        root = ET.fromstring(zf.read(sheet_path))

        parsed_rows = []
        max_col = 0

        for row in root.findall(".//x:sheetData/x:row", NS_MAIN):
            values = {}
            for cell in row.findall("x:c", NS_MAIN):
                ref = cell.attrib.get("r")
                col = _column_index(ref)
                max_col = max(max_col, col)
                cell_type = cell.attrib.get("t")

                value = None
                if cell_type == "inlineStr":
                    parts = [t.text or "" for t in cell.iter("{%s}t" % NS_MAIN["x"])]
                    value = "".join(parts)
                else:
                    v = cell.find("x:v", NS_MAIN)
                    if v is not None:
                        raw = v.text
                        if cell_type == "s":
                            value = shared[int(raw)]
                        elif cell_type in ("str", "e"):
                            value = raw
                        elif cell_type == "b":
                            value = raw == "1"
                        else:
                            try:
                                value = float(raw)
                            except (TypeError, ValueError):
                                value = raw
                values[col] = value

            parsed_rows.append([values.get(i) for i in range(max_col + 1)])

    if not parsed_rows:
        return []

    headers = [str(h) if h is not None else "" for h in parsed_rows[0]]
    records = []
    for row in parsed_rows[1:]:
        padded = row + [None] * (len(headers) - len(row))
        if not any(v is not None for v in padded):
            continue
        records.append(dict(zip(headers, padded)))
    return records


In [3]:

SHEET = "normalization_summary"
REQUIRED = {"mv_method", "outlier_method", "scaler", "feature", "ks_stat"}

cheating_rows = read_xlsx_sheet(CHEATING_FILE, SHEET)
d2_rows = read_xlsx_sheet(D2_FILE, SHEET)

for label, rows in [("Cheating", cheating_rows), ("D2", d2_rows)]:
    if not rows:
        raise ValueError(f"{label}: '{SHEET}' is empty.")
    missing = REQUIRED - set(rows[0])
    if missing:
        raise ValueError(f"{label}: missing required columns: {sorted(missing)}")

print(f"Loaded {len(cheating_rows)} rows from Cheating.")
print(f"Loaded {len(d2_rows)} rows from D2.")


Loaded 48 rows from Cheating.
Loaded 48 rows from D2.


In [4]:

def pipeline_means(rows):
    groups = defaultdict(list)
    for row in rows:
        key = (
            str(row["mv_method"]),
            str(row["outlier_method"]),
            str(row["scaler"]),
        )
        groups[key].append(float(row["ks_stat"]))

    # The current CUMULUS output contains one KS value per feature,
    # i.e. four observations per complete pipeline.
    for key, values in groups.items():
        if len(values) != 4:
            raise ValueError(
                f"Expected 4 feature-level KS values for {key}, found {len(values)}."
            )
    return {key: mean(values) for key, values in groups.items()}

cheating = pipeline_means(cheating_rows)
d2 = pipeline_means(d2_rows)

if set(cheating) != set(d2):
    raise ValueError("Pipeline configurations differ between the two datasets.")

pretty_mv = {"knn(k=5)": "KNN", "locf": "LOCF"}
pretty_out = {"iforest": "Isolation Forest", "mad": "MAD"}
pretty_scaler = {"minmax": "Min–Max", "robust": "Robust", "zscore": "Z-score"}

ranking = []
for key in cheating:
    mv, outlier, scaler = key
    c_ks = cheating[key]
    d_ks = d2[key]
    overall = mean([c_ks, d_ks])

    ranking.append({
        "imputation": pretty_mv.get(mv, mv),
        "outlier_handling": pretty_out.get(outlier, outlier),
        "normalization": pretty_scaler.get(scaler, scaler),
        "cheating_ks": c_ks,
        "d2_ks": d_ks,
        "overall_ks": overall,
    })

ranking.sort(key=lambda r: r["overall_ks"])
for i, row in enumerate(ranking, start=1):
    row["rank"] = i

# Export full ranking
OUTPUT_CSV = BASE / "CUMULUS_pipeline_ranking.csv"
with OUTPUT_CSV.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "rank",
            "imputation",
            "outlier_handling",
            "normalization",
            "cheating_ks",
            "d2_ks",
            "overall_ks",
        ],
    )
    writer.writeheader()
    for row in ranking:
        writer.writerow({
            **row,
            "cheating_ks": f'{row["cheating_ks"]:.9f}',
            "d2_ks": f'{row["d2_ks"]:.9f}',
            "overall_ks": f'{row["overall_ks"]:.9f}',
        })

print(f"Saved: {OUTPUT_CSV.name}")


Saved: CUMULUS_pipeline_ranking.csv


In [5]:

# Display the complete pipeline ranking in the executed notebook.
headers = [
    ("Rank", "rank"),
    ("Imputation", "imputation"),
    ("Outlier handling", "outlier_handling"),
    ("Normalization", "normalization"),
    ("Cheating KS", "cheating_ks"),
    ("D2 KS", "d2_ks"),
    ("Overall KS", "overall_ks"),
]

html = [
    "<table style='border-collapse:collapse;font-family:Arial,sans-serif;font-size:14px'>",
    "<thead><tr>",
]
for title, _ in headers:
    html.append(
        f"<th style='padding:7px 10px;border-bottom:2px solid #555;text-align:left'>{escape(title)}</th>"
    )
html.append("</tr></thead><tbody>")

for row in ranking:
    html.append("<tr>")
    for title, key in headers:
        value = row[key]
        if key.endswith("_ks"):
            text = f"{value:.6f}"
            align = "right"
        elif key == "rank":
            text = str(value)
            align = "right"
        else:
            text = escape(str(value))
            align = "left"
        weight = "font-weight:700;" if row["rank"] == 1 else ""
        html.append(
            f"<td style='padding:6px 10px;border-bottom:1px solid #ddd;"
            f"text-align:{align};{weight}'>{text}</td>"
        )
    html.append("</tr>")

html.append("</tbody></table>")
display(HTML("".join(html)))

best = ranking[0]
print(
    f'\nBest overall pipeline: {best["imputation"]} → '
    f'{best["outlier_handling"]} → {best["normalization"]} '
    f'(overall KS = {best["overall_ks"]:.6f})'
)


Rank,Imputation,Outlier handling,Normalization,Cheating KS,D2 KS,Overall KS
1,KNN,Isolation Forest,Min–Max,0.011640,0.013258,0.012449
2,LOCF,Isolation Forest,Min–Max,0.011761,0.013246,0.012504
3,KNN,MAD,Min–Max,0.012231,0.014748,0.013490
4,LOCF,MAD,Min–Max,0.012367,0.014747,0.013557
5,KNN,Isolation Forest,Robust,0.015515,0.017048,0.016281
6,LOCF,Isolation Forest,Robust,0.015696,0.017025,0.016360
7,KNN,MAD,Robust,0.016287,0.018964,0.017626
8,LOCF,MAD,Robust,0.016502,0.018963,0.017733
9,KNN,Isolation Forest,Z-score,0.022269,0.025592,0.023930
10,LOCF,Isolation Forest,Z-score,0.022594,0.025558,0.024076



Best overall pipeline: KNN → Isolation Forest → Min–Max (overall KS = 0.012449)
